In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install transformers -q

In [5]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import os

# Set seed everywhere for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cu128
GPU available: True


In [6]:
# Download CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

# Download datasets
full_train = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)

# Split train into 45k train / 5k val
train_size = 45000
val_size = 5000
generator = torch.Generator().manual_seed(SEED)
train_set, val_set = torch.utils.data.random_split(
    full_train, [train_size, val_size], generator=generator)

# Create data loaders
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=64, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=64, shuffle=False)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


Train: 45000 | Val: 5000 | Test: 10000


In [7]:
# Save split indices
train_indices = train_set.indices
val_indices   = val_set.indices
np.save('/content/drive/MyDrive/ViT-Robustness/train_indices.npy', train_indices)
np.save('/content/drive/MyDrive/ViT-Robustness/val_indices.npy',   val_indices)
print("Split indices saved to Drive!")

Split indices saved to Drive!
